# RAG chatbot on Vietnamese Wikipedia (UIT-ViQuAD 2.0) — `v3_qwen3`

**Stack:** LangChain + FAISS + `BAAI/bge-m3` embeddings + **Qwen2.5 7B** served by **Ollama**.

**This version:** Change vs best so far: **answer LLM** Qwen2.5 7B -> **Qwen3 8B** (thinking off, Q4_K_M ~5.2 GB, still fits a T4). The judges stay Gemma 2 9B + Qwen2.5 7B so scores stay comparable. Why: once retrieval is fixed, the remaining errors are reading errors (right paragraph, wrong span); Qwen3 reports better multilingual reading comprehension than Qwen2.5 at the same size (Qwen3 Technical Report, 2025).

Overrides vs `src/config.py` defaults: `{"retriever": "hybrid", "reranker": "BAAI/bge-reranker-v2-m3", "llm_model": "qwen3:8b"}`

**What this notebook does**
1. Installs Ollama and pulls the LLM on the Kaggle T4 GPU.
2. Builds a corpus from the 3,171 unique paragraphs of the ViQuAD 2.0 dev set, splits them into chunks and indexes them.
3. Measures retrieval (Hit@k, MRR) on **all 3,852** answerable dev questions.
4. Answers **200 questions** (the original 50 with seed 42, plus 150 more with seed 43) **with RAG** and **without RAG**.
5. Scores answers with EM, F1, Contains and two LLM judges from different families: **Gemma 2 9B** (main: correctness 1-5, faithfulness 1-5) and Qwen2.5 7B (second, to measure judge agreement with Cohen's kappa).

Settings: Kaggle GPU **T4**, Internet **on**, dataset `gamind/uit-viquad-2-0` attached.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q langchain-core langchain-community langchain-text-splitters langchain-huggingface langchain-ollama faiss-cpu rank_bm25 2>&1 | grep -v -i warning | tail -2

## 1. Install Ollama

In [ ]:
!apt-get update -qq > /dev/null && apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama_install.log 2>&1; tail -2 /tmp/ollama_install.log
!ollama --version

## 2. Project code (`src/`)
These cells are generated from the repo's `src/` folder.

In [ ]:
import os
os.makedirs('src', exist_ok=True)

In [ ]:
%%writefile src/__init__.py
# package

In [ ]:
%%writefile src/config.py
"""All settings in one place. Change a value here, not inside the code."""
from dataclasses import dataclass


@dataclass
class Config:
    version: str = "v0_dense"  # results go to results/<version>/

    # data
    viquad_dev: str = "/kaggle/input/uit-viquad-2-0/dev.json"  # Kaggle: gamind/uit-viquad-2-0
    n_questions: int = 200  # answer test set; the first 50 are the original v0 questions
    n_core: int = 50
    seed: int = 42

    # chunking
    chunk_size: int = 500
    chunk_overlap: int = 100
    add_title: bool = True  # prepend the Wikipedia article title to every chunk

    # retrieval
    embed_model: str = "BAAI/bge-m3"
    retriever: str = "dense"  # "dense" (V0) | "hybrid" (V1: BM25 + dense, RRF)
    candidates: int = 20  # hybrid: candidates taken from each retriever before fusion
    rrf_k: int = 60  # RRF constant (Cormack et al. 2009)
    reranker: str = ""  # V2: cross-encoder, e.g. "BAAI/bge-reranker-v2-m3" ("" = off)
    top_k: int = 4
    max_k_curve: int = 10  # Hit@k curve is computed for k = 1..max_k_curve

    # generation (Ollama)
    llm_model: str = "qwen2.5:7b"  # Q4_K_M, ~4.7 GB
    ollama_url: str = "http://localhost:11434"
    temperature: float = 0.0
    num_ctx: int = 4096
    num_predict: int = 128
    think: bool = False  # Qwen3 "thinking" mode; off = short direct answers

    # judges: the main judge is from a DIFFERENT model family than the answer model
    # (avoids self-preference bias); the second judge is used to measure agreement
    judge_model: str = "gemma2:9b"
    second_judge: str = "qwen2.5:7b"

    out_dir: str = "results"

In [ ]:
%%writefile src/data.py
"""Load UIT-ViQuAD 2.0, build the document corpus and the 50-question test set."""
import glob
import json
import os
import random

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter


def find_dev_json(path):
    """Use `path` if it exists, else search /kaggle/input (mount paths differ between images)."""
    if os.path.exists(path):
        return path
    hits = sorted(glob.glob("/kaggle/input/**/dev.json", recursive=True))
    if not hits:
        raise FileNotFoundError(f"{path} not found and no dev.json under /kaggle/input")
    return hits[0]


def load_viquad(path):
    """dev.json is a flat list of {id, title, context, question, answers{text, answer_start}}."""
    path = find_dev_json(path)
    with open(path, encoding="utf-8") as f:
        return json.load(f)["data"]


def build_corpus(records):
    """One Document per unique paragraph. Returns (docs, context -> ctx_id map)."""
    ctx_ids, docs = {}, []
    for r in records:
        if r["context"] not in ctx_ids:
            ctx_ids[r["context"]] = len(docs)
            docs.append(Document(page_content=r["context"],
                                 metadata={"ctx_id": len(docs), "title": r["title"]}))
    return docs, ctx_ids


def answerable_questions(records, ctx_ids):
    """Questions that have a gold answer (ViQuAD 2.0 also has unanswerable ones)."""
    out = []
    for r in records:
        if r["answers"]["text"]:
            out.append({"id": r["id"], "title": r["title"], "question": r["question"],
                        "answers": sorted(set(r["answers"]["text"])),
                        "ctx_id": ctx_ids[r["context"]]})
    return out


def sample_questions(questions, n, seed, n_core=50):
    """First `n_core` = the original 50-question sample (seed), then extra ones (seed + 1).

    Growing the test set this way keeps old results comparable on the first 50 questions.
    """
    qs = sorted(questions, key=lambda q: q["id"])
    core = random.Random(seed).sample(qs, min(n, n_core))
    if n <= n_core:
        return core
    taken = {q["id"] for q in core}
    rest = [q for q in qs if q["id"] not in taken]
    return core + random.Random(seed + 1).sample(rest, n - n_core)


def split_docs(docs, chunk_size, chunk_overlap, add_title=True):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", ", ", " ", ""])
    chunks = splitter.split_documents(docs)
    for i, c in enumerate(chunks):
        c.metadata["chunk_id"] = i
        if add_title:
            c.page_content = f"{c.metadata['title']}: {c.page_content}"
    return chunks

In [ ]:
%%writefile src/metrics.py
"""Answer metrics (SQuAD-style EM/F1 adapted to Vietnamese) and retrieval metrics."""
import re
import string
import unicodedata
from collections import Counter

_PUNCT = set(string.punctuation) | set("“”‘’–—…«»")


def normalize(text):
    text = unicodedata.normalize("NFC", str(text)).lower()
    text = "".join(" " if ch in _PUNCT else ch for ch in text)
    return re.sub(r"\s+", " ", text).strip()


def exact_match(pred, golds):
    p = normalize(pred)
    return float(any(p == normalize(g) for g in golds))


def f1(pred, golds):
    """Token F1 over Vietnamese syllables (space-separated), best over gold answers."""
    p = normalize(pred).split()
    best = 0.0
    for g in golds:
        g = normalize(g).split()
        common = sum((Counter(p) & Counter(g)).values())
        if common:
            pr, rc = common / len(p), common / len(g)
            best = max(best, 2 * pr * rc / (pr + rc))
    return best


def contains(pred, golds):
    """1 if a gold answer appears inside the prediction (LLMs often add a few words)."""
    p = normalize(pred)
    return float(any(normalize(g) in p for g in golds))


def hit_at_k(retrieved_ctx, gold_ctx, k):
    return float(gold_ctx in retrieved_ctx[:k])


def reciprocal_rank(retrieved_ctx, gold_ctx, k):
    for rank, c in enumerate(retrieved_ctx[:k], 1):
        if c == gold_ctx:
            return 1.0 / rank
    return 0.0

In [ ]:
%%writefile src/retrieval.py
"""Retrievers: dense (FAISS) and hybrid (BM25 + FAISS fused with Reciprocal Rank Fusion)."""
import numpy as np
from rank_bm25 import BM25Okapi

from .metrics import normalize


def tokenize(text):
    """Vietnamese syllables, lower-cased, punctuation removed."""
    return normalize(text).split()


class Retriever:
    """search(question, k) -> chunk Documents; search_batch(questions, k) -> chunk index lists.

    mode="dense":  FAISS only (V0).
    mode="hybrid": FAISS top-`candidates` + BM25 top-`candidates`, fused with RRF (V1).
    reranker="<cross-encoder>": re-score the top-`candidates` with a cross-encoder (V2).
    """

    def __init__(self, index, embeddings, chunks, mode="dense", candidates=20, rrf_k=60,
                 reranker="", device="cuda"):
        self.index, self.emb, self.chunks = index, embeddings, chunks
        self.mode, self.candidates, self.rrf_k = mode, candidates, rrf_k
        self.reranker = None
        if reranker:
            from sentence_transformers import CrossEncoder
            self.reranker = CrossEncoder(reranker, max_length=512, device=device)
            if device == "cuda":
                self.reranker.model.half()  # fp16 on GPU: ~2x faster, same ranking
        # FAISS position -> position in `chunks` (same order as from_documents, mapped to be safe)
        store, id_map = index.docstore, index.index_to_docstore_id
        self.pos2chunk = np.array([store.search(id_map[i]).metadata["chunk_id"]
                                   for i in range(index.index.ntotal)])
        self.bm25 = None
        if mode == "hybrid":
            self.bm25 = BM25Okapi([tokenize(c.page_content) for c in chunks])

    def _dense(self, questions, k):
        q = np.asarray(self.emb.embed_documents(questions), dtype="float32")
        _, pos = self.index.index.search(q, k)
        return [[int(self.pos2chunk[p]) for p in row if p >= 0] for row in pos]

    def _bm25(self, question, k):
        scores = self.bm25.get_scores(tokenize(question))
        k = min(k, len(scores))
        top = np.argpartition(-scores, k - 1)[:k]
        return [int(i) for i in top[np.argsort(-scores[top])]]

    def _rrf(self, rankings, k):
        fused = {}
        for ranking in rankings:
            for rank, cid in enumerate(ranking):
                fused[cid] = fused.get(cid, 0.0) + 1.0 / (self.rrf_k + rank + 1)
        return [cid for cid, _ in sorted(fused.items(), key=lambda x: -x[1])[:k]]

    def _first_stage(self, questions, k):
        if self.mode == "dense":
            return self._dense(questions, k)
        n = max(self.candidates, k)
        dense = self._dense(questions, n)
        return [self._rrf([d, self._bm25(q, n)], k) for q, d in zip(questions, dense)]

    def _rerank(self, questions, cand, k):
        pairs = [(q, self.chunks[i].page_content) for q, row in zip(questions, cand) for i in row]
        scores = self.reranker.predict(pairs, batch_size=64, show_progress_bar=False)
        out, j = [], 0
        for row in cand:
            s = scores[j:j + len(row)]
            j += len(row)
            out.append([row[i] for i in np.argsort(-s)[:k]])
        return out

    def search_batch(self, questions, k):
        if self.reranker is None:
            return self._first_stage(questions, k)
        return self._rerank(questions, self._first_stage(questions, max(self.candidates, k)), k)

    def search(self, question, k):
        return [self.chunks[i] for i in self.search_batch([question], k)[0]]

In [ ]:
%%writefile src/rag.py
"""The RAG chatbot: retriever + LLM served by Ollama, wired with LangChain."""
import re

from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama

SYSTEM = ("Bạn là trợ lý hỏi đáp tiếng Việt. Trả lời thật ngắn gọn, chỉ một cụm từ "
          "hoặc một câu ngắn, không giải thích.")

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", SYSTEM + " Chỉ dùng thông tin trong NGỮ CẢNH. "
               "Nếu ngữ cảnh không có câu trả lời, hãy nói: Không biết."),
    ("human", "NGỮ CẢNH:\n{context}\n\nCÂU HỎI: {question}\nTRẢ LỜI:"),
])

NO_RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", SYSTEM + " Nếu không chắc, hãy nói: Không biết."),
    ("human", "CÂU HỎI: {question}\nTRẢ LỜI:"),
])


def get_embeddings(model_name, device="cuda"):
    return HuggingFaceEmbeddings(model_name=model_name, model_kwargs={"device": device},
                                 encode_kwargs={"normalize_embeddings": True, "batch_size": 32})


def build_index(chunks, embeddings):
    # vectors are L2-normalised, so L2 ranking == cosine ranking
    return FAISS.from_documents(chunks, embeddings)


def get_llm(cfg, model=None, json_mode=False):
    model = model or cfg.llm_model
    extra = {"reasoning": cfg.think} if model.startswith("qwen3") else {}  # thinking models only
    return ChatOllama(model=model, base_url=cfg.ollama_url,
                      temperature=cfg.temperature, num_ctx=cfg.num_ctx,
                      num_predict=cfg.num_predict, format="json" if json_mode else None, **extra)


def strip_think(text):
    """Drop a <think>...</think> block if a reasoning model emits one."""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()


def format_context(docs):
    return "\n\n".join(f"[{i + 1}] {d.page_content}" for i, d in enumerate(docs))


class RAGChatbot:
    def __init__(self, retriever, llm, k=4):
        self.retriever, self.k = retriever, k
        self.rag_chain = RAG_PROMPT | llm | StrOutputParser() | RunnableLambda(strip_think)
        self.plain_chain = NO_RAG_PROMPT | llm | StrOutputParser() | RunnableLambda(strip_think)

    def retrieve(self, question, k=None):
        return self.retriever.search(question, k or self.k)

    def answer_with(self, question, docs):
        return self.rag_chain.invoke({"context": format_context(docs), "question": question})

    def ask(self, question):
        docs = self.retrieve(question)
        return {"answer": self.answer_with(question, docs), "docs": docs}

    def ask_no_rag(self, question):
        return {"answer": self.plain_chain.invoke({"question": question}).strip(), "docs": []}

In [ ]:
%%writefile src/judge.py
"""LLM-as-judge with the local model: correctness (vs gold) and faithfulness (vs context)."""
import json
import re

from langchain_core.prompts import ChatPromptTemplate

# Single user message (no system role): works the same for every judge model family.
CORRECTNESS = ChatPromptTemplate.from_messages([
    ("human", "You grade answers to Vietnamese questions. Compare the MODEL ANSWER with the "
               "GOLD ANSWER. Score 1-5: 5 = same meaning as gold, 4 = correct but with extra or "
               "missing minor detail, 3 = partly correct, 2 = mostly wrong, 1 = wrong or "
               "'Không biết'. Reply only JSON: {{\"score\": <1-5>, \"reason\": \"<short>\"}}\n\n"
               "QUESTION: {question}\nGOLD ANSWER: {gold}\nMODEL ANSWER: {answer}"),
])

FAITHFULNESS = ChatPromptTemplate.from_messages([
    ("human", "You check if an answer is supported by the given CONTEXT. Score 1-5: "
               "5 = fully supported, 3 = partly supported, 1 = not supported or invented. "
               "If the answer is 'Không biết', score 5 only if the context really lacks the "
               "answer, else 1. Reply only JSON: {{\"score\": <1-5>, \"reason\": \"<short>\"}}\n\n"
               "CONTEXT:\n{context}\n\nQUESTION: {question}\nANSWER: {answer}"),
])


def parse_score(text):
    try:
        s = int(json.loads(text)["score"])
    except Exception:
        m = re.search(r"[1-5]", text or "")
        s = int(m.group()) if m else 1
    return min(max(s, 1), 5)


class Judge:
    """Scores 1-5. Tries JSON mode, then plain text; returns None if the judge model fails
    (e.g. Ollama aborts a degenerate, repeating generation) so one bad call cannot kill a run."""

    def __init__(self, llm_json, llm_plain=None):
        self.llms = [llm for llm in (llm_json, llm_plain) if llm is not None]
        self.failures = 0

    def _score(self, prompt, inputs):
        for llm in self.llms:
            try:
                return parse_score((prompt | llm).invoke(inputs).content)
            except Exception as e:  # noqa: BLE001 - any backend error -> next attempt
                last = e
        self.failures += 1
        print(f"judge failed: {type(last).__name__}: {str(last)[:120]}")
        return None

    def correctness(self, question, golds, answer):
        return self._score(CORRECTNESS, {"question": question, "gold": " | ".join(golds),
                                         "answer": answer})

    def faithfulness(self, question, context, answer):
        return self._score(FAITHFULNESS, {"question": question, "context": context,
                                          "answer": answer})

In [ ]:
%%writefile src/ollama_utils.py
"""Start a local Ollama server and pull a model (used by the Kaggle notebook)."""
import json
import subprocess
import time
import urllib.request


def server_up(url="http://localhost:11434"):
    try:
        urllib.request.urlopen(f"{url}/api/tags", timeout=2)
        return True
    except Exception:
        return False


def start_server(log_path="ollama.log", url="http://localhost:11434", timeout=60):
    if server_up(url):
        return None
    proc = subprocess.Popen(["ollama", "serve"], stdout=open(log_path, "w"),
                            stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(timeout):
        if server_up(url):
            return proc
        time.sleep(1)
    raise RuntimeError(f"Ollama did not start, see {log_path}")


def pull(model):
    subprocess.run(["ollama", "pull", model], check=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)


def model_info(model, url="http://localhost:11434"):
    req = urllib.request.Request(f"{url}/api/show", data=json.dumps({"model": model}).encode(),
                                 headers={"Content-Type": "application/json"})
    d = json.load(urllib.request.urlopen(req))
    return d.get("details", {})

In [ ]:
%%writefile src/index_store.py
"""Build the FAISS index once (offline, GPU) and load it at serving time.

    python -m src.index_store --viquad_dev data/dev.json --out artifacts/index
"""
import argparse
import json
import os
import time

from langchain_community.vectorstores import FAISS

from .config import Config
from .data import build_corpus, load_viquad, split_docs
from .rag import build_index, get_embeddings


def save(index, cfg, path, extra=None):
    index.save_local(path)
    meta = {"embed_model": cfg.embed_model, "chunk_size": cfg.chunk_size,
            "chunk_overlap": cfg.chunk_overlap, "add_title": cfg.add_title,
            "chunks": index.index.ntotal, **(extra or {})}
    with open(os.path.join(path, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)


def load(path, embeddings):
    """Returns (index, chunks ordered by chunk_id, meta)."""
    index = FAISS.load_local(path, embeddings, allow_dangerous_deserialization=True)  # own file
    docs = [index.docstore.search(i) for i in index.index_to_docstore_id.values()]
    chunks = sorted(docs, key=lambda d: d.metadata["chunk_id"])
    with open(os.path.join(path, "meta.json")) as f:
        meta = json.load(f)
    return index, chunks, meta


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--viquad_dev", default=Config.viquad_dev)
    ap.add_argument("--out", default="artifacts/index")
    ap.add_argument("--device", default="cuda")
    a = ap.parse_args()
    cfg = Config(viquad_dev=a.viquad_dev)
    t = time.time()
    docs, _ = build_corpus(load_viquad(cfg.viquad_dev))
    chunks = split_docs(docs, cfg.chunk_size, cfg.chunk_overlap, cfg.add_title)
    index = build_index(chunks, get_embeddings(cfg.embed_model, a.device))
    save(index, cfg, a.out, {"build_sec": round(time.time() - t, 1)})
    print(f"saved {index.index.ntotal} chunks to {a.out} in {time.time() - t:.0f}s")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/evaluate.py
"""Full pipeline: corpus -> FAISS -> retrieval eval -> RAG vs no-RAG answers -> scores.

Usage (needs a running Ollama server with the model pulled):
    python -m src.evaluate --viquad_dev path/to/dev.json --out_dir results
"""
import argparse
import json
import os
import time

import pandas as pd
from tqdm.auto import tqdm

from .config import Config
from .data import answerable_questions, build_corpus, load_viquad, sample_questions, split_docs
from .judge import Judge
from .metrics import contains, exact_match, f1, hit_at_k, reciprocal_rank
from .rag import RAGChatbot, build_index, format_context, get_embeddings, get_llm
from .retrieval import Retriever


def prepare(cfg):
    records = load_viquad(cfg.viquad_dev)
    docs, ctx_ids = build_corpus(records)
    all_q = answerable_questions(records, ctx_ids)
    test_q = sample_questions(all_q, cfg.n_questions, cfg.seed, cfg.n_core)
    chunks = split_docs(docs, cfg.chunk_size, cfg.chunk_overlap, cfg.add_title)
    stats = {"paragraphs": len(docs), "chunks": len(chunks),
             "answerable_questions": len(all_q), "test_questions": len(test_q)}
    return docs, chunks, all_q, test_q, stats


def retrieval_eval(retriever, chunks, questions, max_k):
    """Hit@k and MRR on ALL answerable dev questions (cheap: no LLM)."""
    t = time.time()
    hits = retriever.search_batch([q["question"] for q in questions], max_k)
    got = [[chunks[i].metadata["ctx_id"] for i in row] for row in hits]
    curve = [{"k": k, "hit": sum(hit_at_k(r, q["ctx_id"], k) for r, q in zip(got, questions))
              / len(questions)} for k in range(1, max_k + 1)]
    mrr = {f"MRR@{k}": sum(reciprocal_rank(r, q["ctx_id"], k) for r, q in zip(got, questions))
           / len(questions) for k in (4, max_k)}
    return pd.DataFrame(curve), mrr, time.time() - t


def make_judges(cfg):
    names = [cfg.judge_model] + ([cfg.second_judge] if cfg.second_judge else [])
    return {n.split(":")[0].replace(".", ""):
            Judge(get_llm(cfg, model=n, json_mode=True), get_llm(cfg, model=n))
            for n in dict.fromkeys(names)}


def answer_eval(cfg, bot, judges, test_q):
    """judges: {name: Judge}; the first one is the main judge (scores + faithfulness)."""
    (main_name, judge), *others = judges.items()
    rows = []
    for q in tqdm(test_q, desc="answering"):
        docs = bot.retrieve(q["question"])
        ctx = format_context(docs)
        retrieved = [d.metadata["ctx_id"] for d in docs]
        t0 = time.time()
        rag = bot.rag_chain.invoke({"context": ctx, "question": q["question"]}).strip()
        t1 = time.time()
        plain = bot.plain_chain.invoke({"question": q["question"]}).strip()
        t2 = time.time()
        rows.append({
            "id": q["id"], "title": q["title"], "question": q["question"],
            "gold": " | ".join(q["answers"]),
            "hit@4": hit_at_k(retrieved, q["ctx_id"], cfg.top_k),
            "rr@4": reciprocal_rank(retrieved, q["ctx_id"], cfg.top_k),
            "rag_answer": rag, "norag_answer": plain,
            "rag_em": exact_match(rag, q["answers"]), "norag_em": exact_match(plain, q["answers"]),
            "rag_f1": f1(rag, q["answers"]), "norag_f1": f1(plain, q["answers"]),
            "rag_contains": contains(rag, q["answers"]),
            "norag_contains": contains(plain, q["answers"]),
            "rag_judge": judge.correctness(q["question"], q["answers"], rag),
            "norag_judge": judge.correctness(q["question"], q["answers"], plain),
            "rag_faithfulness": judge.faithfulness(q["question"], ctx, rag),
            **{f"{mode}_judge_{name}": j.correctness(q["question"], q["answers"], ans)
               for name, j in others for mode, ans in (("rag", rag), ("norag", plain))},
            "rag_sec": round(t1 - t0, 2), "norag_sec": round(t2 - t1, 2),
            "sources": "; ".join(f"{d.metadata['title']}#{d.metadata['ctx_id']}" for d in docs),
        })
    return pd.DataFrame(rows)


def cohen_kappa(a, b):
    """Cohen's kappa for two binary raters (1 = perfect agreement, 0 = chance level)."""
    a, b = a.astype(bool).to_numpy(), b.astype(bool).to_numpy()
    po = (a == b).mean()
    pe = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    return float((po - pe) / (1 - pe)) if pe < 1 else 1.0


def summarize(df):
    s = {"hit@4": df["hit@4"].mean(), "MRR@4": df["rr@4"].mean()}
    for mode in ("rag", "norag"):
        s[mode] = {"EM": df[f"{mode}_em"].mean(), "F1": df[f"{mode}_f1"].mean(),
                   "Contains": df[f"{mode}_contains"].mean(),
                   "Judge(1-5)": df[f"{mode}_judge"].mean(),
                   "Judge>=4": (df[f"{mode}_judge"].dropna() >= 4).mean(),
                   "judge_missing": int(df[f"{mode}_judge"].isna().sum()),
                   "sec/answer": df[f"{mode}_sec"].mean()}
    s["rag"]["Faithfulness(1-5)"] = df["rag_faithfulness"].mean()
    # agreement between the main judge and the second judge (same answers, binary: score >= 4)
    for col in [c for c in df.columns if c.startswith("rag_judge_")]:
        name = col[len("rag_judge_"):]
        pair = pd.DataFrame({"a": pd.concat([df["rag_judge"], df["norag_judge"]]),
                             "b": pd.concat([df[col], df[f"norag_judge_{name}"]])}).dropna()
        a, b = pair["a"] >= 4, pair["b"] >= 4
        s[f"agreement_vs_{name}"] = {"kappa": cohen_kappa(a, b), "agree": (a == b).mean(),
                                    "n": len(pair),
                                    "rag_Judge>=4": (df[col].dropna() >= 4).mean(),
                                    "norag_Judge>=4":
                                        (df[f"norag_judge_{name}"].dropna() >= 4).mean()}
    # RAG answers split by whether retrieval found the gold paragraph
    for hit, name in ((1.0, "rag_when_hit"), (0.0, "rag_when_miss")):
        sub = df[df["hit@4"] == hit]
        s[name] = {"n": len(sub), "F1": sub["rag_f1"].mean() if len(sub) else None,
                   "Judge(1-5)": sub["rag_judge"].mean() if len(sub) else None}
    return json.loads(json.dumps(s, default=float))


def summarize_all(df, n_core=50):
    """Metrics on the original 50 questions and on the full test set."""
    out = {f"test{n_core}": summarize(df.head(n_core))}
    if len(df) > n_core:
        out[f"test{len(df)}"] = summarize(df)
    return out


def run(cfg):
    out = os.path.join(cfg.out_dir, cfg.version)
    os.makedirs(out, exist_ok=True)
    docs, chunks, all_q, test_q, stats = prepare(cfg)
    print("data:", stats)
    with open(os.path.join(out, "questions.json"), "w", encoding="utf-8") as f:
        json.dump(test_q, f, ensure_ascii=False, indent=1)

    t = time.time()
    emb = get_embeddings(cfg.embed_model)
    index = build_index(chunks, emb)
    stats["index_sec"] = round(time.time() - t, 1)

    retriever = Retriever(index, emb, chunks, cfg.retriever, cfg.candidates, cfg.rrf_k,
                          cfg.reranker)
    curve, mrr, sec = retrieval_eval(retriever, chunks, all_q, cfg.max_k_curve)
    curve.to_csv(os.path.join(out, "retrieval_curve.csv"), index=False)

    bot = RAGChatbot(retriever, get_llm(cfg), k=cfg.top_k)
    df = answer_eval(cfg, bot, make_judges(cfg), test_q)
    df.to_csv(os.path.join(out, "predictions.csv"), index=False)

    summary = {"config": vars(cfg), "data": stats,
               "retrieval_all_dev": {**{f"Hit@{r.k}": r.hit for r in curve.itertuples()}, **mrr,
                                     "n": len(all_q), "sec": round(sec, 1)},
               **summarize_all(df, cfg.n_core)}
    with open(os.path.join(out, "metrics.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    return index, emb, bot, df, curve, summary


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    for k, v in vars(Config()).items():
        ap.add_argument(f"--{k}", type=type(v) if not isinstance(v, bool) else lambda x: x == "1",
                        default=v)
    run(Config(**vars(ap.parse_args())))

In [ ]:
%%writefile src/plots.py
"""Figures for the README."""
import os

import matplotlib.pyplot as plt

BLUE, GREY = "#2a6fdb", "#9aa5b1"


def plot_rag_vs_norag(summary, out_dir):
    key = max((k for k in summary if k.startswith("test")), key=lambda k: int(k[4:]))
    s = summary[key]
    names = ["EM", "F1", "Contains", "Judge>=4"]
    rag = [s["rag"][n] * 100 for n in names]
    plain = [s["norag"][n] * 100 for n in names]
    x = range(len(names))
    fig, ax = plt.subplots(figsize=(7, 4))
    c = summary.get("config", {})
    llm = c.get("llm_model", "LLM")
    retr = c.get("retriever", "dense") + (" + rerank" if c.get("reranker") else "")
    b1 = ax.bar([i - 0.2 for i in x], plain, 0.4, label=f"No RAG ({llm} only)", color=GREY)
    b2 = ax.bar([i + 0.2 for i in x], rag, 0.4, label=f"RAG ({retr}, top-{c.get('top_k', 4)})",
                color=BLUE)
    ax.bar_label(b1, fmt="%.0f", fontsize=9)
    ax.bar_label(b2, fmt="%.0f", fontsize=9)
    ax.set_xticks(list(x), names)
    ax.set_ylabel(f"% of {key[4:]} questions")
    ax.set_ylim(0, 105)
    ax.set_title(f"Answer quality on {key[4:]} UIT-ViQuAD 2.0 questions")
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    path = os.path.join(out_dir, "rag_vs_norag.png")
    fig.savefig(path, dpi=150)
    return path


def plot_hit_curve(curve, n, out_dir):
    fig, ax = plt.subplots(figsize=(6, 3.6))
    ax.plot(curve["k"], curve["hit"] * 100, marker="o", color=BLUE)
    for k, h in zip(curve["k"], curve["hit"]):
        if k in (1, 4, curve["k"].max()):
            ax.annotate(f"{h * 100:.1f}", (k, h * 100), textcoords="offset points",
                        xytext=(0, 7), ha="center", fontsize=9)
    ax.set_xlabel("k (retrieved chunks)")
    ax.set_ylabel("Hit@k (%)")
    ax.set_title(f"Retrieval: gold paragraph in top-k ({n} dev questions)")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    path = os.path.join(out_dir, "retrieval_hit_at_k.png")
    fig.savefig(path, dpi=150)
    return path

## 3. Config, start Ollama, pull the LLM

In [ ]:
import time, json
from src.config import Config
from src.ollama_utils import start_server, pull, model_info

cfg = Config(version="v3_qwen3", **{'retriever': 'hybrid', 'reranker': 'BAAI/bge-reranker-v2-m3', 'llm_model': 'qwen3:8b'})
OUT = os.path.join(cfg.out_dir, cfg.version); os.makedirs(OUT, exist_ok=True)
print(cfg)
server = start_server(os.path.join(OUT, "ollama.log"))
for m in dict.fromkeys([cfg.llm_model, cfg.judge_model, cfg.second_judge]):  # judges fixed across versions
    t = time.time(); pull(m)
    print(f"pulled {m} in {time.time() - t:.0f}s:", model_info(m))

## 4. Data: corpus, chunks and the 50 test questions

In [ ]:
import pandas as pd
from src.evaluate import prepare

docs, chunks, all_q, test_q, stats = prepare(cfg)
print(stats)
json.dump(test_q, open(f"{OUT}/questions.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("example chunk:\n", chunks[0].page_content[:300])

## 5. Embeddings (bge-m3) + FAISS index + retriever

In [ ]:
from src.rag import get_embeddings, build_index
from src.retrieval import Retriever

t = time.time()
emb = get_embeddings(cfg.embed_model)
index = build_index(chunks, emb)
retriever = Retriever(index, emb, chunks, cfg.retriever, cfg.candidates, cfg.rrf_k, cfg.reranker)
stats["index_sec"] = round(time.time() - t, 1)
print(f"indexed {index.index.ntotal} chunks in {stats['index_sec']}s, retriever={cfg.retriever}")

# export the index for serving (upload index.tar.gz to a GitHub Release, see README)
from src.index_store import save
save(index, cfg, "index", {"built_by": f"kaggle {cfg.version}"})
!tar -czf {OUT}/index.tar.gz index && rm -rf index && du -h {OUT}/index.tar.gz

## 6. Retrieval quality on all answerable dev questions
A hit means a retrieved chunk comes from the paragraph that holds the gold answer.

In [ ]:
from src.evaluate import retrieval_eval

curve, mrr, sec = retrieval_eval(retriever, chunks, all_q, cfg.max_k_curve)
curve.to_csv(f"{OUT}/retrieval_curve.csv", index=False)
print(f"{len(all_q)} questions in {sec:.0f}s", {k: round(v, 4) for k, v in mrr.items()})
curve.round(4)

## 7. Try the chatbot

In [ ]:
from src.rag import RAGChatbot, get_llm

bot = RAGChatbot(retriever, get_llm(cfg), k=cfg.top_k)
for q in ["Thủ đô của Nhật Bản là thành phố nào?",
          "Bức tường Berlin được xây dựng vào năm nào?",
          test_q[0]["question"]]:
    out = bot.ask(q)
    print("Q:", q)
    print("A:", out["answer"])
    print("sources:", [d.metadata["title"] for d in out["docs"]], "\n")

## 8. Evaluate the test questions: RAG vs no RAG

In [ ]:
from src.evaluate import answer_eval, make_judges, summarize_all

judges = make_judges(cfg)  # main: gemma2 (other family), second: qwen2.5 (agreement check)
print("judges:", list(judges))
t = time.time()
df = answer_eval(cfg, bot, judges, test_q)
print(f"done in {time.time() - t:.0f}s")
df.to_csv(f"{OUT}/predictions.csv", index=False)
df[["question", "gold", "rag_answer", "norag_answer", "hit@4", "rag_judge", "norag_judge"]].head(15)

## 9. Results

In [ ]:
from src.plots import plot_rag_vs_norag, plot_hit_curve

summary = {"config": vars(cfg), "data": stats,
           "retrieval_all_dev": {**{f"Hit@{r.k}": r.hit for r in curve.itertuples()}, **mrr,
                                 "n": len(all_q), "sec": round(sec, 1)},
           **summarize_all(df, cfg.n_core)}
json.dump(summary, open(f"{OUT}/metrics.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
plot_rag_vs_norag(summary, OUT); plot_hit_curve(curve, len(all_q), OUT)

for key in [k for k in summary if k.startswith("test")]:
    t = summary[key]
    print(f"{key}: Hit@4={t['hit@4']:.3f}  MRR@4={t['MRR@4']:.3f}")
    display(pd.DataFrame({m: t[m] for m in ("rag", "norag")}).round(3))

## 10. Save environment info and print results to the log

In [ ]:
import subprocess, sys, importlib.metadata as md_
pk = ["langchain-core", "langchain-community", "langchain-huggingface", "langchain-ollama",
      "faiss-cpu", "rank-bm25", "sentence-transformers", "torch", "transformers"]
env = {p: md_.version(p) for p in pk}
env["python"] = sys.version.split()[0]
env["ollama"] = subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip()
env["gpu"] = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                            capture_output=True, text=True).stdout.strip()
json.dump(env, open(f"{OUT}/environment.json", "w"), indent=2)
print("ENV_JSON", json.dumps(env))
print("METRICS_JSON", json.dumps(summary, ensure_ascii=False))
print("PREDICTIONS_JSON", df.to_json(orient="records", force_ascii=False))